In [12]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE


In [2]:
# Ensure folders exist
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Load dataset
df = pd.read_csv('data/healthcare-dataset-stroke-data.csv')

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
# Drop ID column
df = df.drop('id', axis=1)

# Remove rare 'Other' gender row
df = df[df['gender'] != 'Other']

print(f"Shape after cleaning: {df.shape}")
print(df['gender'].value_counts())

Shape after cleaning: (5109, 11)
gender
Female    2994
Male      2115
Name: count, dtype: int64


In [4]:
# Check missing values
print("Missing values before handling:")
print(df.isnull().sum())

# Fill missing BMI with median
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

print("\nMissing values after handling:")
print(df.isnull().sum())

Missing values before handling:
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

Missing values after handling:
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


In [5]:
# Identify categorical columns
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

# One-hot encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Convert boolean columns to integers if any
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"Encoded dataset shape: {df_encoded.shape}")
df_encoded.head()

Encoded dataset shape: (5109, 16)


,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Male,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.6,1,1,1,0,1,0,0,1,1,0,0
1,61.0,0,0,202.21,28.1,1,0,1,0,0,1,0,0,0,1,0
2,80.0,0,1,105.92,32.5,1,1,1,0,1,0,0,0,0,1,0
3,49.0,0,0,171.23,34.4,1,0,1,0,1,0,0,1,0,0,1
4,79.0,1,0,174.12,24.0,1,0,1,0,0,1,0,0,0,1,0


In [6]:
# Split features and target
X = df_encoded.drop('stroke', axis=1)
y = df_encoded['stroke']

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (5109, 15)
Target shape: (5109,)

Target distribution:
stroke
0    4860
1     249
Name: count, dtype: int64


In [7]:
# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")
print(f"\ny_test distribution:\n{y_test.value_counts()}")

X_train shape: (4087, 15)
X_test shape: (1022, 15)
y_train distribution:
stroke
0    3888
1     199
Name: count, dtype: int64

y_test distribution:
stroke
0    972
1     50
Name: count, dtype: int64


In [13]:
# Standardize features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Save scaler
joblib.dump(scaler, 'models/scaler.pkl')

print(" Scaling complete")
print(" Scaler saved to models/scaler.pkl")

 Scaling complete
 Scaler saved to models/scaler.pkl


In [9]:
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

print(f"\nResampled X_train shape: {X_train_resampled.shape}")

Before SMOTE:
stroke
0    3888
1     199
Name: count, dtype: int64

After SMOTE:
stroke
0    3888
1    3888
Name: count, dtype: int64

Resampled X_train shape: (7776, 15)


In [14]:
# Save processed datasets
X_train_resampled.to_csv('data/X_train.csv', index=False)
X_test_scaled.to_csv('data/X_test.csv', index=False)
pd.Series(y_train_resampled, name='stroke').to_csv('data/y_train.csv', index=False)
pd.Series(y_test, name='stroke').to_csv('data/y_test.csv', index=False)

# Save feature names
joblib.dump(X_train.columns.tolist(), 'models/feature_names.pkl')

print(" Processed files saved successfully")
print("Saved files:")
print("- data/X_train.csv")
print("- data/X_test.csv")
print("- data/y_train.csv")
print("- data/y_test.csv")
print("- models/feature_names.pkl")

 Processed files saved successfully
Saved files:
- data/X_train.csv
- data/X_test.csv
- data/y_train.csv
- data/y_test.csv
- models/feature_names.pkl


In [15]:
print("=" * 60)
print(" PREPROCESSING COMPLETE")
print("=" * 60)

print(f"""
Final dataset summary:

Original dataset size: {len(df)}
Training set size before SMOTE: {len(X_train)}
Training set size after SMOTE: {len(X_train_resampled)}
Test set size: {len(X_test)}

Number of features: {X_train.shape[1]}

Files saved successfully for modeling.
Ready for next step: Model Training
""")

 PREPROCESSING COMPLETE

Final dataset summary:

Original dataset size: 5109
Training set size before SMOTE: 4087
Training set size after SMOTE: 7776
Test set size: 1022

Number of features: 15

Files saved successfully for modeling.
Ready for next step: Model Training

